In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd drive/MyDrive

/content/drive/MyDrive


In [59]:
import torch
import math
from seq2seq_models import preprocess_text, tokenize_data, pad_or_truncate, Vocab, encode, Embedding
from sequential_models import Linear, Tanh, BasicRNN, LSTMCell, GRUCell

Text preprocessing

In [ ]:
dataset_path = "fra.txt"
src, tgt = [], []

with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]

src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

In [ ]:
def masked_softmax(X, valid_lens):
  """
  Mask <pad> tokens
  """
  def _mask(X, valid_len, value=0):
    maxlen = X.shape[-1]
    mask = torch.arange(maxlen) < valid_len.unsqueeze(-1)
    X[~mask] = value
    return X

  if valid_lens is None:
    return torch.nn.functional.softmax(X, dim=-1)
  else:
    if valid_lens.dim() == 1:
      valid_lens = torch.repeat_interleave(valid_lens, X.shape[1])
    else:
      valid_lens = valid_lens.reshape(-1)

    X = _mask(X, valid_lens, value=-1e6)
    return torch.nn.functional.softmax(X, dim=-1)


In [ ]:
class DotProductAttention():
  def __init__(self):
    pass

  def __repr__(self):
    return f"DotProductAttention()"

  def __call__(self, queries, keys, values, valid_lens=None):
    d = queries.shape[-1]
    scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
    weights = masked_softmax(scores, valid_lens)
    return torch.bmm(weights, values)

In [ ]:
class AdditiveAttention():
  def __init__(self, n_hidden, batch_size):
    #shape of keys and values = [seq_len, batch_size, n_hidden]
    #hidden state of decoder(query) = [batch_size, n_hidden]
    self.Wk = Linear(batch_size, n_hidden)
    self.Wq = Linear(batch_size, n_hidden)
    self.Wv = Linear(n_hidden, 1)
    self.tanh = Tanh()

  def __repr__(self):
    return f"AdditiveAttention()"

  def __call__(self, keys, queries, values, valid_lens):
    features = self.Wk(keys) + self.Wq(queries)
    scores = self.Wv.T @ self.tanh(features)
    self.weights = masked_softmax(scores, valid_lens)
    return torch.bmm(self.weights, values)

  def parameters(self):
    return [self.Wk, self.Wq, self.Wv]